In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH, MODELS

time = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_PATH = MODELS / "dino_adapter_block/20260801_221150"

EMBED_PATH = EMBEDS_DIR / "dino/pretrained"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"
RESULTS_DIR = RESULTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidCover, EllipsoidEvaluator, EllipsoidFitter, CandidateCleaner
from src.algorithims.ellipsoid.bootstrap import BootstrapRunner
from src.types import ExperimentConfig, AlgorithmResults

from src.stats.mahalanobis_detector import MahalanobisDetector

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

REG = 1e-4
fitter = EllipsoidFitter(support_points=5, reg=REG)
cleaner = CandidateCleaner(fitter=fitter, min_points=1)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)
evaluator = EllipsoidEvaluator(reg=REG)

mal_detector = MahalanobisDetector(reg=1e-6) # smaller reg as its a larger matrix

In [6]:
with open(MODEL_PATH / "metadata.json", "r") as f:
    model_metadata = json.load(f)
    
neg_indices = model_metadata.get("negative_indices", [])

In [7]:
test_df : list[pd.DataFrame] = []
train_df: list[pd.DataFrame] = []

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category 
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[
        (meta["split"] != "train")
        & (~meta.index.isin(neg_indices))
    ]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[
        (meta["split"] != "train")
        & (~meta.index.isin(neg_indices))
    ]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    runner = BootstrapRunner(
        cover=cover,
        evaluator=evaluator,
        mal_detector=mal_detector,
        n_test_bootstraps=1000,
        n_train_bootstraps=1000,
        seed=42,
    )

    test_bootstraps, train_bootstraps = runner.run(
        train_emb=cat_emb, 
        good_test_emb=good_test_emb, 
        defect_test_emb=defect_test_emb
        )

    train_bootstraps.insert(0, "category", category)
    test_bootstraps.insert(0, "category", category)

    train_bootstraps.to_csv(RESULTS_DIR / f"{category}_train_bootstraps.csv", index=False)
    test_bootstraps.to_csv(RESULTS_DIR / f"{category}_test_bootstraps.csv", index=False)

    test_df.append(test_bootstraps)
    train_df.append(train_bootstraps)

Running bottle


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Running cable


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Running zipper


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Train Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
train_summaries: list[pd.DataFrame] = []
test_summaries: list[pd.DataFrame] = []

for category, train_bootstraps, test_bootstraps in zip(
    categories,
    train_df,
    test_df,
):
    train_summary = runner.summarise_bootstrap(train_bootstraps)
    test_summary = runner.summarise_bootstrap(test_bootstraps)

    # In case summarise_bootstrap returns a Series
    if isinstance(train_summary, pd.Series):
        train_summary = train_summary.to_frame().T

    if isinstance(test_summary, pd.Series):
        test_summary = test_summary.to_frame().T

    train_summary.insert(0, "category", category)
    test_summary.insert(0, "category", category)

    train_summaries.append(train_summary)
    test_summaries.append(test_summary)

train_summary_df = pd.concat(
    train_summaries,
    ignore_index=True,
)

test_summary_df = pd.concat(
    test_summaries,
    ignore_index=True,
)

In [9]:
train_summary_df.to_csv(
    RESULTS_DIR / "train_bootstrap_summary.csv",
    index=False,
)

train_summary_df

,category,auroc_mean,auroc_std,auroc_ci_lower,auroc_ci_upper,best_threshold_mean,best_threshold_std,best_threshold_ci_lower,best_threshold_ci_upper,accuracy_mean,...,pc95_median_ci_lower,pc95_median_ci_upper,pc1_ratio_mean_mean,pc1_ratio_mean_std,pc1_ratio_mean_ci_lower,pc1_ratio_mean_ci_upper,rank_mean_mean,rank_mean_std,rank_mean_ci_lower,rank_mean_ci_upper
0,bottle,0.995344,0.003841,0.988095,1.000000,6.264019e+05,64424.020066,4.935577e+05,7.692612e+05,0.965952,...,4.0,5.0,0.565309,0.020788,0.525691,0.607486,5.343534,0.576462,4.438763,6.595601
1,cable,0.889037,0.012191,0.864121,0.912861,2.566008e+06,169745.351853,2.249119e+06,2.902859e+06,0.802327,...,4.0,5.0,0.519456,0.019927,0.485190,0.561456,5.552864,0.577625,4.619048,6.928571
2,capsule,0.869794,0.017783,0.834454,0.903480,4.585573e+05,38765.144640,3.861298e+05,5.366536e+05,0.814364,...,4.0,5.0,0.541581,0.020400,0.500671,0.582339,5.493654,0.571581,4.523810,6.744629
3,carpet,0.969640,0.004601,0.960273,0.978341,8.783294e+05,74118.828566,7.476579e+05,1.032548e+06,0.916419,...,4.0,5.0,0.535904,0.017741,0.501508,0.570842,6.402845,0.591606,5.391304,7.808511
4,grid,0.954035,0.008678,0.936487,0.969089,1.350218e+06,178219.733223,9.553379e+05,1.697595e+06,0.872872,...,4.0,4.0,0.586883,0.018295,0.552007,0.622861,6.178816,0.583749,5.195205,7.489710
5,hazelnut,0.922584,0.008573,0.904412,0.938854,2.421431e+06,153734.236273,2.130143e+06,2.737325e+06,0.855000,...,5.0,6.5,0.474775,0.015371,0.445896,0.506403,7.781156,0.512990,6.811230,8.849528
6,leather,0.999909,0.000272,0.998981,1.000000,1.529825e+06,138721.542713,1.281034e+06,1.821436e+06,0.990435,...,4.0,5.0,0.554511,0.020189,0.514913,0.596224,5.863742,0.549956,4.931779,7.136843
7,metal_nut,0.935846,0.012050,0.911034,0.957967,8.990819e+05,69072.688980,7.776172e+05,1.049138e+06,0.859678,...,4.0,5.0,0.536965,0.020561,0.496965,0.579315,5.581342,0.585840,4.593772,6.809635
8,pill,0.888542,0.013266,0.861975,0.914082,1.169895e+06,129024.718595,9.216022e+05,1.384384e+06,0.776018,...,4.0,6.0,0.505518,0.018585,0.468271,0.540926,6.115908,0.507194,5.260459,7.217512
9,screw,0.795249,0.019799,0.754453,0.831538,6.830707e+05,55367.922169,5.699099e+05,7.873634e+05,0.765350,...,4.0,5.0,0.528802,0.016773,0.497614,0.564125,6.949032,0.593558,5.895376,8.306803


In [10]:
test_summary_df.to_csv(
    RESULTS_DIR / "test_bootstrap_summary.csv",
    index=False,
)

test_summary_df

,category,auroc_mean,auroc_std,auroc_ci_lower,auroc_ci_upper,best_threshold_mean,best_threshold_std,best_threshold_ci_lower,best_threshold_ci_upper,accuracy_mean,...,max_winning_fraction_ci_lower,max_winning_fraction_ci_upper,mal_auroc_mean,mal_auroc_std,mal_auroc_ci_lower,mal_auroc_ci_upper,delta_mean,delta_std,delta_ci_lower,delta_ci_upper
0,bottle,0.996833,0.003560,0.987302,1.000000,6.077094e+05,72290.066161,4.414650e+05,7.352861e+05,0.962096,...,0.132530,0.253012,1.000000,1.756295e-17,1.000000,1.000000,-0.003167,0.003560,-0.012698,0.000000
1,cable,0.882135,0.025322,0.831522,0.927104,2.316533e+06,177679.963187,1.827184e+06,2.615614e+06,0.795313,...,0.093333,0.166667,0.945444,1.732591e-02,0.909478,0.976204,-0.063309,0.019045,-0.102708,-0.028673
2,capsule,0.873011,0.037432,0.793777,0.938183,4.094893e+05,29668.656413,3.412645e+05,5.165189e+05,0.801735,...,0.075758,0.143939,0.946538,2.138822e-02,0.899462,0.980864,-0.073527,0.027543,-0.128470,-0.027513
3,carpet,0.972699,0.014365,0.939797,0.994783,7.078472e+05,30036.247789,6.611892e+05,7.442832e+05,0.934778,...,0.119658,0.222222,0.985292,9.468397e-03,0.961477,0.999197,-0.012593,0.006570,-0.026495,-0.002408
4,grid,0.970481,0.015550,0.931495,0.994152,1.034772e+06,79938.016877,7.055432e+05,1.084895e+06,0.905256,...,0.141026,0.307692,0.981020,1.679614e-02,0.941500,1.000000,-0.010539,0.015597,-0.045134,0.020050
5,hazelnut,0.930890,0.023801,0.881956,0.971759,1.974290e+06,143711.570012,1.861867e+06,2.325364e+06,0.872443,...,0.122642,0.235849,0.962954,1.692397e-02,0.924913,0.989551,-0.032064,0.021081,-0.076654,0.006966
6,leather,0.999654,0.000585,0.997962,1.000000,1.281423e+06,95008.896619,1.154517e+06,1.539486e+06,0.981984,...,0.475806,0.612903,1.000000,0.000000e+00,1.000000,1.000000,-0.000346,0.000585,-0.002038,0.000000
7,metal_nut,0.934621,0.022288,0.886119,0.972642,8.358563e+05,41845.994456,7.109383e+05,8.769003e+05,0.851626,...,0.086957,0.147826,0.982566,9.783400e-03,0.960411,0.997556,-0.047945,0.016343,-0.082625,-0.020528
8,pill,0.909241,0.025387,0.853239,0.954746,9.850421e+05,83148.988924,8.393164e+05,1.113231e+06,0.792491,...,0.083832,0.155689,0.950123,1.853004e-02,0.907242,0.979555,-0.040882,0.021509,-0.084322,-0.000805
9,screw,0.807644,0.041473,0.725143,0.886268,5.362721e+05,50345.016834,4.730195e+05,6.966617e+05,0.784744,...,0.056250,0.112500,0.822077,4.021523e-02,0.739076,0.895071,-0.014434,0.022640,-0.057409,0.029125


In [ ]:
test_summary_df["auroc_mean"].mean(), test_summary_df["mal_auroc_mean"].mean(), test_summary_df["delta_mean"].mean()

np.float64(-0.027310172255928425)

In [16]:
train_summary_df["auroc_mean"].mean(), train_summary_df["mal_auroc_mean"].mean(), train_summary_df["delta_mean"].mean()

(np.float64(0.9241285539339938),
 np.float64(0.9565820918094502),
 np.float64(-0.03245353787545641))

In [13]:
runner.avg_ell_eval_time, runner.avg_mal_eval_time, runner.avg_ell_eval_time - runner.avg_mal_eval_time

(0.010287745620998066, 0.0015907773749913759, 0.00869696824600669)

In [14]:
runner.avg_ell_fit_time, runner.avg_mal_fit_time, runner.avg_ell_fit_time - runner.avg_mal_fit_time

(0.1145300124609921, 0.01018984902801094, 0.10434016343298116)